In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.notebook import tqdm

sns.set(style='whitegrid', palette='viridis')

In [2]:
import os

BASE_DIR = '../plots/zad3' 
plot_iters_dir = os.path.join(BASE_DIR, 'wykresy_iteracji') 
os.makedirs(plot_iters_dir, exist_ok=True) 

plot_dir = os.path.join(BASE_DIR, 'wykresy_boxplot')
os.makedirs(plot_dir, exist_ok=True)
stats_dir = os.path.join(BASE_DIR, 'statystyki_metryki')
os.makedirs(stats_dir, exist_ok=True)
print(f"Przygotowano foldery w: {BASE_DIR}")

Przygotowano foldery w: ../plots/zad3


In [3]:
def find_mle_logistic(xs, sigma, start_theta, max_iter=100, tol=1e-6):
    theta_k = float(start_theta) 
    n = len(xs)

    for k in range(max_iter):
        # --- Obliczanie F(x_i) ---
        z = (xs - theta_k) / sigma
        exp_minus_z = np.exp(-z)
        F_x = 1.0 / (1.0 + exp_minus_z)

        # --- Obliczanie l'(theta) ---
        l_prime = (1.0 / sigma) * np.sum(2.0 * F_x - 1.0)

        # --- Obliczanie l''(theta) ---
        F_times_1_minus_F = exp_minus_z / (1.0 + exp_minus_z)**2
        l_double_prime = (-2.0 / sigma**2) * np.sum(F_times_1_minus_F)

        # --- Krok Newtona-Raphsona ---
        
        if abs(l_double_prime) < 1e-10:
            return theta_k, k+1

        step = l_prime / l_double_prime
        theta_k_plus_1 = theta_k - step

        if abs(step) < tol:
            return theta_k_plus_1, k + 1 # Zbieżność osiągnięta

        theta_k = theta_k_plus_1

    return theta_k, max_iter # Nie zbiegło

In [4]:
ns = [20, 50, 100]
scenarios = {
    'a': {'theta': 0.0, 'sigma': 1.0},
    'b': {'theta': 0.0, 'sigma': 2.0},
    'c': {'theta': 4.0, 'sigma': 1.0},
    'd': {'theta': 4.0, 'sigma': 2.0},
}
T = 10**4 

start_strategies = {
    'Mediana': lambda xs: np.median(xs),
    'Średnia': lambda xs: np.mean(xs),
    'Stała 0.0': lambda xs: 0.0,
    'Maksimum': lambda xs: np.max(xs),
}

results = [] 
simulations = {} 

for n in ns:
    simulations[n] = {}
    for name, params in scenarios.items():
        true_theta = params['theta']
        true_sigma = params['sigma']
        
        samples = [np.random.logistic(loc=true_theta, scale=true_sigma, size=n) 
                    for _ in range(T)]
        simulations[n][(true_theta, true_sigma)] = samples

        scenario_label = f"({name}) θ={true_theta}, σ={true_sigma}"
        
        for start_name, start_func in start_strategies.items():
            
            for i in tqdm(range(T), desc=f"n={n}, Scen. {name}, Start: {start_name}"):
                
                xs = simulations[n][(true_theta, true_sigma)][i]
                
                start_theta = start_func(xs)
                
                theta_hat, moves = find_mle_logistic(xs, sigma=true_sigma, start_theta=start_theta)

                results.append({
                    'n': n,
                    'true_theta': true_theta,
                    'true_sigma': true_sigma,
                    'theta_hat': theta_hat,
                    'moves': moves,
                    'start_strategy': start_name,
                    'scenario': scenario_label
                })

df_results = pd.DataFrame(results)

print(f"Liczba zebranych wyników: {len(df_results)}")
print("Pierwsze 5 wierszy danych:")
print(df_results.head())

n=20, Scen. a, Start: Mediana:   0%|          | 0/10000 [00:00<?, ?it/s]

n=20, Scen. a, Start: Średnia:   0%|          | 0/10000 [00:00<?, ?it/s]

n=20, Scen. a, Start: Stała 0.0:   0%|          | 0/10000 [00:00<?, ?it/s]

n=20, Scen. a, Start: Maksimum:   0%|          | 0/10000 [00:00<?, ?it/s]

/tmp/ipykernel_58030/2995181690.py:8: RuntimeWarning: overflow encountered in exp
  exp_minus_z = np.exp(-z)
/tmp/ipykernel_58030/2995181690.py:15: RuntimeWarning: invalid value encountered in divide
  F_times_1_minus_F = exp_minus_z / (1.0 + exp_minus_z)**2
/tmp/ipykernel_58030/2995181690.py:15: RuntimeWarning: overflow encountered in square
  F_times_1_minus_F = exp_minus_z / (1.0 + exp_minus_z)**2


...

n=100, Scen. d, Start: Mediana:   0%|          | 0/10000 [00:00<?, ?it/s]

n=100, Scen. d, Start: Średnia:   0%|          | 0/10000 [00:00<?, ?it/s]

n=100, Scen. d, Start: Stała 0.0:   0%|          | 0/10000 [00:00<?, ?it/s]

n=100, Scen. d, Start: Maksimum:   0%|          | 0/10000 [00:00<?, ?it/s]

Liczba zebranych wyników: 480000
Pierwsze 5 wierszy danych:
    n  true_theta  true_sigma  theta_hat  moves start_strategy  \
0  20         0.0         1.0  -0.324185      4        Mediana   
1  20         0.0         1.0  -0.746792      3        Mediana   
2  20         0.0         1.0  -0.184038      3        Mediana   
3  20         0.0         1.0   0.555275      3        Mediana   
4  20         0.0         1.0  -0.012554      3        Mediana   

           scenario  
0  (a) θ=0.0, σ=1.0  
1  (a) θ=0.0, σ=1.0  
2  (a) θ=0.0, σ=1.0  
3  (a) θ=0.0, σ=1.0  
4  (a) θ=0.0, σ=1.0  


In [5]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(14, 11))

scenarios_list = list(scenarios.items()) 
axes_list = axes.flat 

for (name, params), ax in zip(scenarios_list, axes_list):

    true_theta = params['theta']
    true_sigma = params['sigma']

    scenario_label = f"({name}) $\\theta={true_theta}, \\sigma={true_sigma}$"

    # FILTROWANIE: Używamy tylko wyników z głównej strategii (Mediana)
    data_for_plot = df_results[
        (df_results['true_theta'] == true_theta) &
        (df_results['true_sigma'] == true_sigma) &
        (df_results['start_strategy'] == 'Mediana') 
    ]

    sns.boxplot(
        data=data_for_plot,
        x='n',
        y='theta_hat',
        ax=ax,
        showfliers=False 
    )
    ax.axhline(
        true_theta,
        color='red',
        linestyle='--',
        label=f'Prawdziwe $\\theta = {true_theta}$'
    )

    ax.set_title(scenario_label, fontsize=14)
    ax.set_xlabel('Rozmiar próby (n)', fontsize=12)
    ax.set_ylabel('Oszacowana wartość $\\hat{\\theta}$', fontsize=12)
    ax.legend()

fig.suptitle('Porównanie Estymatorów $\\hat{\\theta}$ (MLE) dla Rozkładu Logistycznego (Start: Mediana)',
             fontsize=18)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

filename = os.path.join(plot_dir, 'mle_logistic_boxplot_comparison.png')
plt.savefig(filename)
plt.close() 
print(f"Zapisano Boxploty do: {filename}")

Zapisano Boxploty do: ../plots/zad3/wykresy_boxplot/mle_logistic_boxplot_comparison.png


In [8]:
print("\n--- 1. Metryki Jakości Estymatora (Tylko Mediana) ---")

df_mediana = df_results[df_results['start_strategy'] == 'Mediana'].copy()

grouped_mediana = df_mediana.groupby(['n', 'true_theta', 'true_sigma'])

bias = grouped_mediana['theta_hat'].mean() - grouped_mediana['true_theta'].first()
variance = grouped_mediana['theta_hat'].var()
mse = ((df_mediana['theta_hat'] - df_mediana['true_theta'])**2).groupby(
    [df_mediana['n'], df_mediana['true_theta'], df_mediana['true_sigma']]
).mean()

df_metrics = pd.DataFrame({
    'n': [idx[0] for idx in bias.index],
    'true_theta': [idx[1] for idx in bias.index],
    'true_sigma': [idx[2] for idx in bias.index],
    'Bias': bias.values,
    'Variance': variance.values,
    'MSE': mse.values
})

df_metrics[['Bias', 'Variance', 'MSE']] = df_metrics[['Bias', 'Variance', 'MSE']].round(5)
metrics_filename = os.path.join(stats_dir, 'podsumowanie_metryk_mle_logistic_MEDIANA.csv')
df_metrics.to_csv(metrics_filename, index=False)
print(f"CSV z głównymi metrykami (Mediana) zapisany: {metrics_filename}")
print(df_metrics)


print("\n--- 2. Średnia Liczba Kroków (Efektywność Algorytmu) ---")

df_moves = (
    df_results
    .groupby(['start_strategy'])['moves']
    .mean()
    .reset_index()
)

df_moves = df_moves.round(3)

moves_filename = os.path.join(stats_dir, 'srednia_liczba_krokow.csv')
df_moves.to_csv(moves_filename, index=False)
print(f"CSV ze średnią liczbą kroków zapisany: {moves_filename}")

print(df_moves)


--- 1. Metryki Jakości Estymatora (Tylko Mediana) ---
CSV z głównymi metrykami (Mediana) zapisany: ../plots/zad3/statystyki_metryki/podsumowanie_metryk_mle_logistic_MEDIANA.csv
      n  true_theta  true_sigma     Bias  Variance      MSE
0    20         0.0         1.0 -0.00106   0.15090  0.15089
1    20         0.0         2.0  0.01249   0.59641  0.59651
2    20         4.0         1.0 -0.00277   0.15287  0.15287
3    20         4.0         2.0 -0.00685   0.61512  0.61511
4    50         0.0         1.0  0.00129   0.06055  0.06055
5    50         0.0         2.0  0.00177   0.24028  0.24026
6    50         4.0         1.0 -0.00349   0.05959  0.05959
7    50         4.0         2.0 -0.00670   0.24201  0.24203
8   100         0.0         1.0  0.00039   0.02940  0.02940
9   100         0.0         2.0 -0.00531   0.11856  0.11858
10  100         4.0         1.0  0.00104   0.02993  0.02993
11  100         4.0         2.0  0.00815   0.12104  0.12109

--- 2. Średnia Liczba Kroków (Efektywność